# 02 - Bronze Delta to Silver Delta

Use Spark SQL to type, validate, deduplicate, and combine batch and streaming events. Invalid records become a separate quarantine Delta table.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.delta_pipeline import DeltaSettings, build_silver, create_delta_spark_session, delta_history

settings = DeltaSettings.from_env()
spark = create_delta_spark_session("InsightsStream-Silver-Delta", settings)

In [ ]:
silver_rows, rejected_rows = build_silver(spark, settings)
print(f"silver rows={silver_rows}, quarantined rows={rejected_rows}")

silver = spark.read.format("delta").load(settings.uri("silver/events"))
silver.createOrReplaceTempView("silver_events")
spark.sql("""
    SELECT source_type, metric, COUNT(*) AS clean_events, ROUND(SUM(value), 2) AS total_value
    FROM silver_events
    GROUP BY source_type, metric
    ORDER BY source_type, clean_events DESC
""").show(truncate=False)
delta_history(spark, "silver/events", settings).select("version", "timestamp", "operation").show(truncate=False)